# Data Science Salary Analysis - Data Exploration

This notebook explores salary data for data science roles across Canadian provinces to help students make informed co-op and career decisions.

## Objectives
1. Explore salary distributions across provinces
2. Calculate cost-of-living adjusted salaries
3. Identify key insights for students
4. Generate visualizations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. Data Collection

Data sources:
- Government of Canada Job Bank (NOC 21211: Data Scientists)
- LinkedIn Salary Insights
- Glassdoor Canada Salary Reports

In [ ]:
# Salary data by province (in thousands CAD)
provinces = ['BC', 'ON', 'AB', 'SK', 'QC', 'MB']
median_salaries = [141, 129, 114, 112, 110, 94]
low_salaries = [83, 80, 79, 78, 77, 75]
high_salaries = [239, 221, 229, 198, 201, 181]

# Cost of living index (Vancouver = 100)
# Based on Numbeo Cost of Living Index 2025
cost_of_living = {
    'BC': 100,      # Vancouver baseline
    'ON': 85,       # Toronto
    'AB': 72,       # Calgary
    'SK': 68,       # Saskatoon
    'QC': 75,       # Montreal
    'MB': 70        # Winnipeg
}

# Create DataFrame
df = pd.DataFrame({
    'Province': provinces,
    'Median_Salary': median_salaries,
    'Low_Salary': low_salaries,
    'High_Salary': high_salaries,
    'CoL_Index': [cost_of_living[p] for p in provinces]
})

# Calculate salary ranges
df['Salary_Range'] = df['High_Salary'] - df['Low_Salary']

print("Raw Salary Data:")
df

## 2. Cost-of-Living Adjustment

Adjusting salaries for cost of living gives a more accurate picture of purchasing power.

In [ ]:
# Calculate adjusted salaries (normalized to Vancouver cost of living)
df['Adjusted_Median'] = (df['Median_Salary'] / df['CoL_Index']) * 100
df['Adjusted_Low'] = (df['Low_Salary'] / df['CoL_Index']) * 100
df['Adjusted_High'] = (df['High_Salary'] / df['CoL_Index']) * 100

print("\nCost-of-Living Adjusted Salaries:")
df[['Province', 'Median_Salary', 'Adjusted_Median', 'CoL_Index']].round(1)

## 3. Key Insights

In [ ]:
# Find best provinces by adjusted salary
df_sorted = df.sort_values('Adjusted_Median', ascending=False)

print("\n🏆 TOP PROVINCES BY PURCHASING POWER:")
print("="*60)
for idx, row in df_sorted.iterrows():
    print(f"{row['Province']}: ${row['Median_Salary']}k nominal → ${row['Adjusted_Median']:.0f}k adjusted")
    print(f"   (Cost of living: {row['CoL_Index']}% of Vancouver)\n")

# Calculate percentage differences
print("\n📊 KEY STATISTICS:")
print("="*60)
print(f"Highest nominal salary: {df['Median_Salary'].max()}k ({df.loc[df['Median_Salary'].idxmax(), 'Province']})")
print(f"Highest adjusted salary: {df['Adjusted_Median'].max():.0f}k ({df.loc[df['Adjusted_Median'].idxmax(), 'Province']})")
print(f"\nSalary range spread: {df['Median_Salary'].max() - df['Median_Salary'].min()}k difference")
print(f"Average entry-level: ${df['Low_Salary'].mean():.0f}k")
print(f"Average senior-level: ${df['High_Salary'].mean():.0f}k")

## 4. Visualizations

In [ ]:
# Comparison: Nominal vs Adjusted Salaries
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Nominal salaries
colors = ['#3498DB' if p == 'BC' else '#E74C3C' if p == 'AB' else '#95A5A6' for p in df['Province']]
ax1.bar(df['Province'], df['Median_Salary'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Nominal Median Salaries', fontsize=14, fontweight='bold')
ax1.set_ylabel('Annual Salary ($k CAD)', fontsize=11)
ax1.set_xlabel('Province', fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, (p, v) in enumerate(zip(df['Province'], df['Median_Salary'])):
    ax1.text(i, v + 2, f'${v}k', ha='center', fontsize=10, fontweight='bold')

# Adjusted salaries
ax2.bar(df['Province'], df['Adjusted_Median'], color=colors, edgecolor='black', linewidth=1.5)
ax2.set_title('Cost-of-Living Adjusted Salaries', fontsize=14, fontweight='bold')
ax2.set_ylabel('Adjusted Salary ($k CAD)', fontsize=11)
ax2.set_xlabel('Province', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, (p, v) in enumerate(zip(df['Province'], df['Adjusted_Median'])):
    ax2.text(i, v + 2, f'${v:.0f}k', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../output/salary_comparison_adjusted.png', dpi=300, bbox_inches='tight')
plt.show()

print("💡 Notice how rankings change when cost of living is factored in!")

In [ ]:
# Career progression analysis
fig, ax = plt.subplots(figsize=(12, 7))

x_pos = np.arange(len(provinces))
width = 0.6

# Plot ranges
for i, (p, low, med, high) in enumerate(zip(df['Province'], df['Low_Salary'], 
                                              df['Median_Salary'], df['High_Salary'])):
    color = '#3498DB' if p == 'BC' else '#E74C3C' if p == 'AB' else '#95A5A6'
    linewidth = 4 if p in ['BC', 'AB'] else 2
    
    # Draw range line
    ax.plot([i, i], [low, high], color=color, linewidth=linewidth, alpha=0.8)
    
    # Mark median
    ax.scatter(i, med, color=color, s=150, zorder=5, edgecolor='black', linewidth=1.5)
    
    # Add growth potential annotation
    growth = ((high - low) / low) * 100
    ax.text(i, high + 5, f'+{growth:.0f}%', ha='center', fontsize=9, style='italic')

ax.set_xticks(x_pos)
ax.set_xticklabels(provinces)
ax.set_ylabel('Annual Salary ($k CAD)', fontsize=12)
ax.set_xlabel('Province', fontsize=12)
ax.set_title('Career Growth Potential: Entry-Level to Senior Positions', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#3498DB', linewidth=4, label='BC (Highest nominal)'),
    Line2D([0], [0], color='#E74C3C', linewidth=4, label='AB (Best value)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
           markersize=10, label='Median salary', markeredgecolor='black')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('../output/career_growth_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Student Recommendations

Based on the analysis:

In [ ]:
print("\n🎯 RECOMMENDATIONS FOR STUDENTS:")
print("="*70)

top_adjusted = df_sorted.iloc[0]
print(f"\n1️⃣ BEST VALUE: {top_adjusted['Province']}")
print(f"   - Adjusted salary: ${top_adjusted['Adjusted_Median']:.0f}k")
print(f"   - Nominal salary: ${top_adjusted['Median_Salary']}k")
print(f"   - Cost of living: {top_adjusted['CoL_Index']}% of Vancouver")
print(f"   → Best purchasing power for your money!")

top_nominal = df.loc[df['Median_Salary'].idxmax()]
print(f"\n2️⃣ HIGHEST SALARY: {top_nominal['Province']}")
print(f"   - Nominal salary: ${top_nominal['Median_Salary']}k")
print(f"   - Entry to senior: ${top_nominal['Low_Salary']}k → ${top_nominal['High_Salary']}k")
print(f"   → Best for resume building in major tech hub")

# Entry-level expectations
avg_entry = df['Low_Salary'].mean()
print(f"\n3️⃣ CO-OP SALARY EXPECTATIONS:")
print(f"   - Entry-level range: ${df['Low_Salary'].min()}k - ${df['Low_Salary'].max()}k annually")
print(f"   - Hourly equivalent: ${df['Low_Salary'].min()*1000/2080:.0f} - ${df['Low_Salary'].max()*1000/2080:.0f}/hour")
print(f"   → Aim for $35-40/hour for data science co-ops")

print("\n" + "="*70)

## 6. Export Results

In [ ]:
# Save analysis results to CSV
output_df = df[['Province', 'Median_Salary', 'Low_Salary', 'High_Salary', 
                 'CoL_Index', 'Adjusted_Median', 'Salary_Range']].round(1)

output_df.to_csv('../data/salary_analysis_results.csv', index=False)
print("✅ Results saved to data/salary_analysis_results.csv")

# Display summary
output_df.sort_values('Adjusted_Median', ascending=False)